In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType

# Players data
players_data = [
    (15, 1),
    (25, 1),
    (30, 1),
    (45, 1),
    (10, 2),
    (35, 2),
    (50, 2),
    (20, 3),
    (40, 3)
]

players_schema = StructType([
    StructField("player_id", IntegerType(), True),
    StructField("group_id", IntegerType(), True)
])

players_df = spark.createDataFrame(players_data, schema=players_schema)
players_df.createOrReplaceTempView("players")
display(players_df)

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType

matches_data = [
    (1, 15, 45, 3, 0),
    (2, 30, 25, 1, 2),
    (3, 30, 15, 2, 0),
    (4, 40, 20, 5, 2),
    (5, 35, 50, 1, 1)
]

matches_schema = StructType([
    StructField("match_id", IntegerType(), True),
    StructField("first_player", IntegerType(), True),
    StructField("second_player", IntegerType(), True),
    StructField("first_score", IntegerType(), True),
    StructField("second_score", IntegerType(), True)
])

matches_df = spark.createDataFrame(matches_data, schema=matches_schema)
matches_df.createOrReplaceTempView("matches")
display(matches_df)

In [0]:
spark.sql(
    """
        with cte as (
        select p.*, m.score from players p join 
        (select m.player, sum(m.score) from
        (select first_player as player, first_score as score from matches 
        union
        select second_player as player, second_score as score from matches) m
        group by m.player
        on p.player_id = m.player)

        select * from cte
        group by player
    """
).display()

In [0]:
spark.sql("""
WITH player_scores AS (
    SELECT first_player AS player_id, first_score AS score FROM matches
    UNION ALL
    SELECT second_player AS player_id, second_score AS score FROM matches
), 
final_scores AS (
    SELECT p.group_id, ps.player_id, SUM(score) AS score
    FROM player_scores ps
    INNER JOIN players p ON p.player_id=ps.player_id
    GROUP BY p.group_id, ps.player_id
), 
final_ranking AS (
    SELECT *
    , RANK() OVER(PARTITION BY group_id ORDER BY score DESC , player_id ASC) AS rn
    FROM final_scores
)

SELECT * FROM final_ranking WHERE rn=1
""").display()